# 05 RAG Foundations

By the end you can index travel docs, retrieve scored chunks, and ground an agent answer in sources.


Set up the project path and reload shared helpers from this repo.


In [2]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "shared").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# Pick up edits to shared/ without a full kernel restart (re-run this cell)
for _name in (
    "shared.dataflow",
    "shared.notebook_display",
    "shared.llm",
    "shared.bootcamp_fixtures",
):
    importlib.reload(importlib.import_module(_name))

print(f"Project root: {ROOT}")


Project root: c:\Users\Azooo\langchain-bootcamp


Check which API keys and provider settings are available in `.env`.


In [3]:
import os

def env_status():
    keys = {
        "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
        "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
        "LLM_PROVIDER": os.getenv("LLM_PROVIDER", "deepseek"),
        "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
        "LANGSMITH_API_KEY": bool(os.getenv("LANGSMITH_API_KEY")),
    }
    for k, v in keys.items():
        print(f"{k}: {v}")
    return keys

ENV = env_status()


DEEPSEEK_API_KEY: True
DEEPSEEK_MODEL: deepseek-v4-flash
LLM_PROVIDER: deepseek
TAVILY_API_KEY: True
LANGSMITH_API_KEY: True


Load the chat model helper used by live cells below.


In [4]:
import os
from shared.llm import get_model

def require_llm():
    return get_model()


Import DATAFLOW printers once, they label each agent and RAG step so you read a run like a wizard tracing a spell. Later cells call print_agent_dataflow or print_rag_dataflow; watch those blocks closely.


In [ ]:
from shared.dataflow import preview, print_agent_dataflow, print_rag_dataflow, print_final_state

Load real travel docs, split, embed, and store in memory. RAG starts with chunk count, first run downloads the embedding model. Print should show docs count, chunk count, and a sample source filename.


In [6]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from shared.bootcamp_fixtures import load_travel_corpus

docs = load_travel_corpus(ROOT)
print("Sample source:", docs[0].metadata.get("source", "?")[:60])

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
vs = InMemoryVectorStore.from_documents(chunks, embeddings)
print(f"{len(docs)} docs to {len(chunks)} chunks")
print("Sample chunk:", chunks[0].page_content[:120], "...")
print("Source:", Path(chunks[0].metadata.get("source", "")).name)


Loaded 9 documents from data/
Sample source: c:\Users\Azooo\langchain-bootcamp\data\eu_air_passenger_righ


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3882.93it/s]


9 docs to 998 chunks
Sample chunk: SOURCE: https://transport.ec.europa.eu/transport-themes/passenger-rights/air_en
LICENSE: European Union official informa ...
Source: eu_air_passenger_rights.txt


Define search_knowledge_base with scores, see what retrieval returns before generation. Chunk IDs and similarity scores tell you if the right policy text was found.


In [ ]:
from langchain_core.tools import tool

@tool(response_format="content_and_artifact")
def search_knowledge_base(query: str):
    """Search travel knowledge base: Wikipedia, TSA, EU rights, State Dept advisories."""
    hits = vs.similarity_search_with_score(query, k=3)
    parts, artifacts = [], []
    for i, (doc, score) in enumerate(hits, 1):
        src = Path(doc.metadata.get("source", "")).name
        parts.append(f"[{i}:{src} score={score:.3f}]\n{doc.page_content[:350]}")
        artifacts.append(doc)
    return "\n---\n".join(parts), artifacts

content = search_knowledge_base.invoke({"query": "Schengen visa 90 days"})
print(content)


[1:wikipedia_schengen_visa.txt score=0.793]
The entry requirements for third country nationals who intend to stay in the Schengen Area for not more than 90 days in any 180-day period are as follows:
---
[2:wikipedia_schengen_visa.txt score=0.752]
==== Common Schengen Visa Policy ====

The common visa policy allows nationals of certain countries to enter the Schengen Area via air, land or sea without a visa for stays of up to 90 days within a 180-day period. Nationals of certain other countries are required to have a visa either upon arrival or in transit.


== Current members ==
---
[3:wikipedia_schengen_visa.txt score=0.750]
90 days but no more than one year. If a Schengen state wishes to allow the holder of a long-stay visa to remain there for longer than a year, the state must issue him or her with a residence permit.


In [8]:
from langchain.agents import create_agent

rag_agent = create_agent(model=require_llm(), tools=[search_knowledge_base],
    system_prompt="Cite knowledge base sources for policy questions.")
question = "What are EU air passenger delay compensation rules?"
retrieved = search_knowledge_base.invoke({"query": "EU passenger delay compensation"})
print_rag_dataflow(question, retrieved)
ans = rag_agent.invoke({"messages": [{"role": "user", "content": question}]})
print_agent_dataflow(ans["messages"])


DATAFLOW
1. question: What are EU air passenger delay compensation rules?
2. retrieve: [1:eu_air_passenger_rights.txt score=0.736] Communication COM(2007)168 final  of the Commission to the European Parliament and the Council pursuant to Article 17 of Regulation (EC)...
DATAFLOW
[0] HumanMessage: What are EU air passenger delay compensation rules?
[1] AIMessage tool_calls: ["search_knowledge_base({'query': 'EU air passenger delay compensation rules'})"]
[2] ToolMessage(search_knowledge_base): [1:eu_air_passenger_rights.txt score=0.772] Communication COM(2007)168 final  of the Commission to the European Parliament and the Council pursuant to Article 17 of Regulation (EC)...
[3] AIMessage tool_calls: ["search_knowledge_base({'query': 'EU Regulation 261/2004 compensation amounts delay'})"]
[4] ToolMessage(search_knowledge_base): [1:eu_air_passenger_rights.txt score=0.628] Communication COM(2007)168 final  of the Commission to the European Parliament and the Council pursuant to Article 17 